# W&B Registry — Publish the HiLift demo dataset

The **W&B Registry** is an organisation-level catalogue of
promotable artifacts. It sits on top of project-scoped artifacts:
you `log_artifact` inside a project as usual, then `link_artifact`
into a Registry **Collection** to make the artifact org-visible,
aliasable (`production`, `staging`, ...), and discoverable across
projects.

The workshop's Registry is called **`hilift`**. This notebook adds
a **`pointcloud_demo`** Collection to it, containing the
demo-synthetic slat / main-wing / flap point clouds that
`experiment_tracking_hilift.ipynb` (and the downstream sweep and
report notebooks) consume.

The demo data stands in for the real NASA CRM-HL boundary VTU
files. Real HiLiftAeroML boundary archives are ~13 GB per case,
which is impractical for a workshop laptop. The demo generator
produces point clouds with the same tensor shapes as the real data,
so the training code in the downstream notebook runs unchanged if
you later swap in real HiLift boundary caches.

**What this notebook does**
1. Generate a small demo-synthetic point-cloud dataset (12 cases).
2. Serialise each case to a compressed NPZ file.
3. Log the collection as a project-level artifact, then
   `link_artifact` it into the Registry's `pointcloud_demo`
   Collection with `production` + `v1` aliases. The `hilift`
   Registry is created via `api.create_registry(...)` inside
   `try/except` if it doesn't yet exist.
4. Read the dataset back from the Registry as a verification step.

**Prerequisites**
- `uv sync`
- `uv run wandb login`
- Access to the `wandb_emea` entity — or edit `ENTITY` in §1.

**Run order in the workshop**
`registry.ipynb` (this notebook) → `experiment_tracking_hilift.ipynb`
→ `sweeps.ipynb` → `reports.ipynb`.

**Idempotence.** `create_registry` is wrapped in `try/except` and
`log_artifact` versions monotonically. Re-running this notebook
publishes a new version and moves the `:production` alias to it;
nothing breaks.


## 1. Setup

Third-party imports and the section-local constants.


In [ ]:
from __future__ import annotations

import tempfile
from pathlib import Path

import numpy as np
import wandb


In [ ]:
# ---- W&B target ----
ENTITY = "wandb_emea"
PROJECT = "jb_physics_demo"

# ---- Registry destination ----
REGISTRY_NAME = "hilift"
DATASET_COLLECTION = "pointcloud_demo"
DATASET_ARTIFACT_NAME = "hilift_pointcloud_demo"

# ---- Demo dataset parameters ----
# 12 cases → the training notebook's 70/15/15 split yields 8 train /
# 1 val / 3 test, which lets `TABLE_SAMPLE_SIZE = 3` (in that notebook)
# draw a genuine sample of distinct test cases each cadence epoch.
SEED = 42
CASE_SPECS = [
    ("LHC001",  4), ("LHC001",  8), ("LHC001", 12), ("LHC001", 16),
    ("LHC002",  4), ("LHC002",  8), ("LHC002", 12), ("LHC002", 16),
    ("LHC003",  4), ("LHC003",  8), ("LHC003", 12), ("LHC003", 16),
]
POINTS_PER_CASE_CACHE = 8192

# ---- Local scratch ----
TMP_DIR = Path(tempfile.mkdtemp(prefix="hilift_registry_"))

print(f"wandb          = {wandb.__version__}")
print(f"target project = {ENTITY}/{PROJECT}")
print(f"Registry path  = wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}")
print(f"tmp            = {TMP_DIR}")


## 2. The demo point-cloud generator

The real HiLiftAeroML boundary VTU files are ~13 GB each, which
knocks them out of scope for a laptop workshop. This notebook
substitutes a small synthetic generator that keeps the important
properties for pipeline testing while dropping the disk cost.

**Geometry.** Each case is a point cloud arranged as three
components in roughly the CRM-HL proportions:

- **Slat** (~22 % of points, ahead of the leading edge)
- **Main wing** (~56 % of points, along the chord)
- **Flap** (~22 % of points, behind the main wing, deflected by
  an angle that depends on AoA and a per-geometry phase)

Points are placed in `(x, y, z)` with spanwise `y ∈ [-3.2, 3.2]`
and chord/thickness shapes chosen so the resulting cloud looks
qualitatively like a high-lift wing.

**Target field.** A scalar `synthetic_pressure_like_cp` combining
four physically-motivated pieces:

1. **Leading-edge suction** — a Gaussian dip whose depth scales
   with angle of attack.
2. **Flap loading** — a localised negative-pressure region behind
   the main wing, weighted by the flap mask.
3. **Slat loading** — analogous localised suction on the slat.
4. **Pressure recovery** — a mild linear increase in `x` and `y`
   plus a per-geometry offset.

Plus a small Gaussian noise term for realism.

**Same tensor shape as the real HiLift caches** —
`points (N, 3)`, `normals (N, 3)`, `target (N, 1)` — so the exact
training code in the downstream notebook runs unchanged if you
later swap in real data.

**Caveat.** The synthetic Cp is not aerodynamically valid. Use it
to shake out the pipeline, not to make surrogate-quality claims.


In [ ]:
def _run_id(geo_id: str, aoa: float | int) -> str:
    """Canonical run/case identifier — matches the reference notebook's shape."""
    return f"geo_{geo_id}_AoA_{int(aoa)}"


def create_demo_pointcloud_case(
    geo_id: str,
    aoa: float | int,
    n_points: int,
    seed: int,
) -> dict:
    """Build one synthetic slat/main-wing/flap point-cloud case.

    Returns a dict with the same keys the real HiLift boundary
    caches expose, so downstream code is agnostic to source.
    """
    rid = f"demo_{_run_id(geo_id, aoa)}"
    rng_seed = seed + int(float(aoa) * 10) + sum(ord(ch) for ch in geo_id)
    rng = np.random.default_rng(rng_seed)
    n = n_points

    # ---- Component assignment ----
    component = rng.choice(3, size=n, p=[0.22, 0.56, 0.22])  # slat, main, flap

    y = rng.uniform(-1.0, 1.0, size=n)
    span = np.abs(y)
    chord = 1.0 - 0.32 * span
    sweep = 0.22 * span

    geo_num = int("".join(ch for ch in geo_id if ch.isdigit()) or 1)
    geo_phase = 0.5 + 0.5 * np.sin(geo_num * 1.7)

    x = np.zeros(n, dtype=np.float32)
    z = np.zeros(n, dtype=np.float32)
    normals = np.zeros((n, 3), dtype=np.float32)

    slat = component == 0
    main = component == 1
    flap = component == 2

    # ---- Component-specific coordinates and normals ----
    # Main wing: along the chord, gentle top-surface arch.
    u = rng.uniform(0.04, 0.96, size=main.sum())
    x[main] = sweep[main] + u * chord[main]
    z[main] = 0.055 * np.sin(np.pi * u) * (1.0 - 0.25 * span[main])
    normals[main] = np.array([0.0, 0.0, 1.0], dtype=np.float32)

    # Slat: ahead of the LE, slight leading-edge droop.
    u = rng.uniform(-0.20, 0.05, size=slat.sum())
    x[slat] = sweep[slat] + u * chord[slat]
    z[slat] = 0.075 + 0.025 * np.cos((u + 0.20) / 0.25 * np.pi) * (1.0 - 0.2 * span[slat])
    normals[slat] = np.array([-0.20, 0.0, 0.98], dtype=np.float32)

    # Flap: behind the main wing, deflected by AoA + per-geometry phase.
    u = rng.uniform(0.90, 1.25, size=flap.sum())
    flap_deflection = np.deg2rad(22.0 + 0.35 * float(aoa) + 7.0 * geo_phase)
    x[flap] = sweep[flap] + u * chord[flap]
    z[flap] = -0.035 - np.sin(flap_deflection) * (u - 0.90) * 0.55
    z[flap] += 0.025 * np.sin(np.pi * np.clip((u - 0.90) / 0.35, 0, 1)) * (1.0 - 0.25 * span[flap])
    normals[flap] = np.array(
        [np.sin(flap_deflection), 0.0, np.cos(flap_deflection)],
        dtype=np.float32,
    )

    points = np.stack([x, 3.2 * y, z], axis=1).astype(np.float32)
    normals /= np.maximum(np.linalg.norm(normals, axis=1, keepdims=True), 1e-6)

    # ---- Pressure-like target field ----
    aoa_scale = float(aoa) / 16.0
    leading_edge = sweep + 0.08 * chord

    le_suction = (
        -(0.65 + 0.75 * aoa_scale)
        * np.exp(-((x - leading_edge) ** 2) / 0.012)
        * (1.0 - 0.28 * span)
    )
    flap_loading = (
        -0.45
        * np.exp(-((x - (sweep + 1.03 * chord)) ** 2) / 0.035)
        * flap.astype(float)
    )
    slat_loading = (
        -0.38
        * np.exp(-((x - (sweep - 0.08 * chord)) ** 2) / 0.018)
        * slat.astype(float)
    )
    pressure_recovery = 0.18 * x + 0.035 * y + 0.06 * geo_phase
    noise = rng.normal(0.0, 0.015, size=n)

    target = (
        le_suction + flap_loading + slat_loading + pressure_recovery + noise
    ).astype(np.float32)[:, None]

    # ---- Assemble result ----
    return {
        "points": points,
        "normals": normals.astype(np.float32),
        "target": target,
        "geo_id": geo_id,
        "aoa": float(aoa),
        "run_id": rid,
        "target_name": "synthetic_pressure_like_cp",
        "target_location": "generated_demo",
        "target_mode": "scalar",
    }


## 3. Generate the dataset

We generate one case per `(geo_id, aoa)` in `CASE_SPECS` and write
each to its own compressed NPZ file. One file per case makes the
downstream consumer (the training notebook) easy: it lists the
artifact directory and loads each NPZ independently.


In [ ]:
demo_cases: list[dict] = []
case_files: list[Path] = []

for i, (geo_id, aoa) in enumerate(CASE_SPECS):
    case = create_demo_pointcloud_case(
        geo_id=geo_id,
        aoa=aoa,
        n_points=POINTS_PER_CASE_CACHE,
        seed=SEED,
    )
    demo_cases.append(case)

    case_path = TMP_DIR / f"case_{i:03d}_{case['run_id']}.npz"
    np.savez_compressed(
        case_path,
        points=case["points"],
        normals=case["normals"],
        target=case["target"],
        geo_id=np.array(case["geo_id"]),
        aoa=np.array(case["aoa"], dtype=np.float32),
        run_id=np.array(case["run_id"]),
        target_name=np.array(case["target_name"]),
        target_location=np.array(case["target_location"]),
        target_mode=np.array(case["target_mode"]),
    )
    case_files.append(case_path)

    print(
        f"generated {case['run_id']:32s} "
        f"points={case['points'].shape}   "
        f"target=({case['target'].min():+.3f}, {case['target'].max():+.3f})"
    )

print()
print(f"wrote {len(case_files)} NPZ files to {TMP_DIR}")


## 4. Log + promote to the Registry

The two-step pattern:

1. `run.log_artifact(...)` — creates a **project-level** artifact
   (versioned, but only visible inside this project).
2. `run.link_artifact(target_path="wandb-registry-{R}/{C}", aliases=[...])`
   — promotes that specific version into the Registry Collection.
   Org-visible, aliased, cross-project consumable. The Collection
   is created implicitly on first link; the Registry itself is
   created via `api.create_registry(...)` inside a `try/except` so
   re-runs are a no-op.

`log_artifact` is asynchronous — we call `.wait()` on the returned
reference before linking so the alias resolves on the backend.


In [ ]:
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    job_type="dataset-registration",
    name="hilift-pointcloud-demo-register",
    tags=["notebook", "hilift", "dataset"],
) as run:
    api = wandb.Api()

    # ---- Ensure the Registry exists (idempotent) ----
    try:
        api.create_registry(name=REGISTRY_NAME, visibility="organization")
        print(f"created registry: {REGISTRY_NAME}")
    except Exception as exc:
        print(f"registry {REGISTRY_NAME} create skipped: {exc}")

    # ---- Assemble the dataset artifact ----
    art = wandb.Artifact(
        name=DATASET_ARTIFACT_NAME,
        type="dataset",
        description=(
            "Demo-synthetic slat/main-wing/flap point clouds with a "
            "pressure-like Cp target. Stand-in for real CRM-HL boundary "
            "fields; identical schema so the same training code runs "
            "against either."
        ),
        metadata={
            "n_cases": len(demo_cases),
            "points_per_case": POINTS_PER_CASE_CACHE,
            "case_specs": [list(cs) for cs in CASE_SPECS],
            "target_name": "synthetic_pressure_like_cp",
            "target_mode": "scalar",
            "seed": SEED,
        },
    )
    for path in case_files:
        art.add_file(str(path), name=path.name)

    # ---- Log + link ----
    logged = run.log_artifact(art)
    logged.wait()                                  # ensure upload commits

    target_path = f"wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}"
    run.link_artifact(
        logged,
        target_path=target_path,
        aliases=["production", "v1"],
    )

    print()
    print(f"published -> {target_path}:production")
    print(f"run URL   -> {run.url}")


## 5. Verify — read the dataset back from the Registry

Sanity-check the round trip: a fresh downstream run consumes the
promoted entry. This is exactly what `experiment_tracking_hilift.ipynb`
does at its own setup step — so if this cell works end-to-end, the
training notebook will find the dataset without further plumbing.


In [ ]:
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    job_type="registry-verify",
    name="hilift-dataset-verify",
    tags=["notebook", "hilift", "registry", "verify"],
) as verify_run:
    art = verify_run.use_artifact(
        f"wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}:production"
    )
    download_dir = Path(art.download())
    npz_files = sorted(download_dir.glob("*.npz"))

    print(f"pulled {len(npz_files)} case NPZs from the Registry")

    # Load the first case as a shape/scalar sanity check.
    first = np.load(npz_files[0], allow_pickle=True)
    print()
    print(f"first case: {npz_files[0].name}")
    print(f"  geo_id      = {str(first['geo_id'])}")
    print(f"  aoa         = {float(first['aoa']):.1f}")
    print(f"  points      shape = {tuple(first['points'].shape)}")
    print(f"  normals     shape = {tuple(first['normals'].shape)}")
    print(f"  target      shape = {tuple(first['target'].shape)}   "
          f"(range: {first['target'].min():+.3f} .. {first['target'].max():+.3f})")


## 6. Wrap up — what to look at in the W&B UI

**Runs** (two)
- `hilift-pointcloud-demo-register` (`job_type = dataset-registration`)
- `hilift-dataset-verify` (`job_type = registry-verify`)

**Registry**
- `wandb-registry-hilift/pointcloud_demo` now contains one version
  of `hilift_pointcloud_demo`, aliased `production` + `v1`.

**Lineage graph**
- `hilift_pointcloud_demo:v0` produced by the register run and
  consumed by the verify run.

**Next step.** Open `notebooks/experiment_tracking_hilift.ipynb`.
That notebook picks up where this one left off — it consumes the
dataset at `wandb-registry-hilift/pointcloud_demo:production`,
trains a Transolver++ point-field surrogate on it, and promotes
the best-val checkpoint into a `checkpoints` Collection under the
same `hilift` Registry.
